# 🖼️ AI 이미지 생성 — 포스터, 앨범 커버, 무대 컨셉을 만들어보세요

텍스트로 이미지를 만들거나, 기존 이미지를 새로운 스타일로 변형할 수 있습니다.
공연 포스터, 앨범 커버, 무대 컨셉 이미지 등에 활용해보세요.

아래 세 가지 도구 중 **원하는 것만 골라서** 실행하세요.

| 도구 | 입력 | 특징 | 다운로드 |
|------|------|------|---------|
| **SD-Turbo** | 텍스트 | 1스텝 초고속 생성 | ~3.4GB (빠름) |
| **SDXL-Turbo** | 텍스트 | 더 높은 품질 | ~6.5GB |
| **SD-Turbo img2img** | 이미지 + 텍스트 | 기존 이미지를 스타일 변형 | (위 모델 공유) |

▶ 먼저 공통 패키지를 설치합니다.

⚠️ **GPU 설정 확인**: 상단 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU** 를 선택해주세요.
GPU가 설정되지 않으면 실행이 매우 느리거나 에러가 납니다.

In [ ]:
!pip install -q diffusers transformers accelerate

import warnings
warnings.filterwarnings('ignore')

print('✅ 공통 패키지 설치 완료!')

---
# 옵션 A: SD-Turbo — 1스텝 초고속 이미지 생성

**SD-Turbo**는 단 **1스텝**으로 이미지를 생성하는 가장 빠른 모델입니다.
모델 다운로드도 ~3.4GB로 가볍습니다.

⏰ 모델 다운로드 ~1분 + 이미지 생성 약 1초

### A-1. 모델 로드

▶ 아래 셀을 실행하세요.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

print('🔄 SD-Turbo 모델을 불러오는 중... (~3.4GB 다운로드)')

# SD-Turbo: 860M params, 1-step 생성, 가장 빠름
model_id = 'stabilityai/sd-turbo'

# 더 높은 품질을 원하면 아래로 교체 (~6.5GB 다운로드):
# model_id = 'stabilityai/sdxl-turbo'

# 최고 품질을 원하면 (느리지만 정교함, ~10GB 다운로드):
# from diffusers import StableDiffusion3Pipeline
# pipe = StableDiffusion3Pipeline.from_pretrained('stabilityai/stable-diffusion-3.5-medium', torch_dtype=torch.float16)
# pipe.enable_model_cpu_offload()

pipe_txt = AutoPipelineForText2Image.from_pretrained(
    model_id, torch_dtype=torch.float16, variant='fp16'
)
pipe_txt = pipe_txt.to('cuda')

print(f'✅ {model_id.split("/")[1]} 로딩 완료!')

### A-2. 프롬프트 입력 → 이미지 생성

▶ 만들고 싶은 이미지를 영어로 설명하세요.

**프롬프트 예시:**

| 프롬프트 | 느낌 |
|---------|------|
| `minimalist concert poster, solo pianist on dark stage, single spotlight` | 미니멀 공연 포스터 |
| `watercolor painting of piano keys transforming into ocean waves` | 수채화 앨범 커버 |
| `abstract digital art inspired by Chopin nocturne, deep blue and gold` | 추상 디지털 아트 |

In [ ]:
import matplotlib.pyplot as plt
import os

prompt = "minimalist concert poster, solo pianist on dark stage, single spotlight"  # ← 여기를 수정하세요

print(f'🎨 프롬프트: "{prompt}"')
print('🔄 이미지를 생성하고 있습니다...')

image = pipe_txt(
    prompt=prompt,
    num_inference_steps=1,   # SD-Turbo: 1스텝
    guidance_scale=0.0,      # SD-Turbo: guidance 불필요
    width=512, height=512,
).images[0]

output_file = 'generated_image.png'
image.save(output_file)

print('✅ 생성 완료!')
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis('off')
plt.title(prompt[:60] + '...', fontsize=10)
plt.show()

from google.colab import files
files.download(output_file)

---
# 옵션 B: SDXL-Turbo — 더 높은 품질

**SDXL-Turbo**는 SD-Turbo보다 큰 모델 (2.6B params)로, **2스텝**에 더 높은 품질을 냅니다.
다운로드가 ~6.5GB로 좀 더 오래 걸립니다.

⏰ 모델 다운로드 ~2분 + 이미지 생성 약 2초

### B-1. 모델 로드 → 프롬프트 입력 → 생성

▶ 프롬프트를 수정하고 셀을 실행하세요. 모델 로딩에 1~2분 걸립니다.

In [ ]:
import torch
import gc
import matplotlib.pyplot as plt

# 이전 모델 메모리 해제
if 'pipe_txt' in dir():
    del pipe_txt
gc.collect()
torch.cuda.empty_cache()

print('🔄 SDXL-Turbo 모델을 불러오는 중... (~6.5GB 다운로드)')

from diffusers import AutoPipelineForText2Image

# SDXL-Turbo: 2.6B params, 더 높은 품질
# 최고 품질을 원하면 SD 3.5 Medium으로 교체 (~10GB, 28스텝 필요):
# from diffusers import StableDiffusion3Pipeline
# pipe_xl = StableDiffusion3Pipeline.from_pretrained('stabilityai/stable-diffusion-3.5-medium', torch_dtype=torch.float16)
# pipe_xl.enable_model_cpu_offload()

pipe_xl = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16'
)
pipe_xl = pipe_xl.to('cuda')
print('✅ SDXL-Turbo 로딩 완료!')

# === 여기를 수정하세요 ===
prompt = "watercolor painting of piano keys transforming into ocean waves"  # ← 프롬프트
# ========================

print(f'🎨 프롬프트: "{prompt}"')
print('🔄 이미지를 생성하고 있습니다...')

image = pipe_xl(
    prompt=prompt,
    num_inference_steps=2,
    guidance_scale=0.0,
    width=512, height=512,
).images[0]

output_file = 'sdxl_output.png'
image.save(output_file)

print('✅ 생성 완료!')
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis('off')
plt.show()

from google.colab import files
files.download(output_file)

---
# 옵션 C: SD-Turbo img2img — 기존 이미지를 새로운 스타일로

본인의 사진이나 그림을 넣고, AI가 **단 1스텝**만에 새로운 스타일로 변형합니다.

**핵심 파라미터 — `strength`:**
- `0.3` → 원본이 거의 유지됨 (살짝 변형)
- `0.7` → 원본 구도는 남지만 스타일 변형
- `1.0` → 원본 거의 무시, 완전 새로 생성

⏰ 이미지 변형에 약 1초

### C-1. 이미지 업로드

▶ 변형할 이미지를 업로드하세요.

In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

uploaded = files.upload()
if not uploaded:
    print('⚠️ 파일이 선택되지 않았습니다. 다음 셀에서 예시 파일을 사용하세요.')
else:
    input_image_path = list(uploaded.keys())[0]

input_image = Image.open(input_image_path).convert('RGB').resize((512, 512))

    print(f'✅ 이미지 업로드: {input_image_path}')
plt.figure(figsize=(6, 6))
plt.imshow(input_image)
plt.axis('off')
plt.title('원본 이미지')
plt.show()

### C-2. 스타일 변형

▶ 프롬프트와 strength를 수정하고 실행하세요.

In [ ]:
import torch
import gc
from diffusers import AutoPipelineForImage2Image
import matplotlib.pyplot as plt

# 이전 모델 메모리 해제
for name in ['pipe_txt', 'pipe_xl']:
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

print('🔄 SD-Turbo img2img 모델을 불러오는 중...')

# SD-Turbo: 가볍고 빠름 (~3.4GB)
model_id = 'stabilityai/sd-turbo'
# 더 높은 품질을 원하면:
# model_id = 'stabilityai/sdxl-turbo'  # ~6.5GB

pipe_img2img = AutoPipelineForImage2Image.from_pretrained(
    model_id, torch_dtype=torch.float16, variant='fp16',
)
pipe_img2img = pipe_img2img.to('cuda')
print(f'✅ {model_id.split("/")[1]} img2img 로딩 완료!')

# === 여기를 수정하세요 ===
prompt = "watercolor painting, artistic, beautiful colors"  # ← 원하는 스타일
strength = 0.7  # ← 0~1 사이. 높을수록 크게 변형
# ========================

print(f'🎨 프롬프트: "{prompt}" (strength: {strength})')
print('🔄 스타일 변형 중...')

result = pipe_img2img(
    prompt=prompt,
    image=input_image,
    strength=strength,
    guidance_scale=0.0,
    num_inference_steps=1,
).images[0]

output_file = 'img2img_output.png'
result.save(output_file)

print('✅ 변형 완료!')

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(input_image)
axes[0].set_title('원본')
axes[0].axis('off')
axes[1].imshow(result)
axes[1].set_title(f'변형 (strength={strength})')
axes[1].axis('off')
plt.tight_layout()
plt.show()

from google.colab import files
files.download(output_file)